# Reproducible test evaluation in Colab

Install one reviewed commit, connect Drive, select available systems, then run and inspect each system's fixed ten-item development pilot before its 395-item test. Every system retains both generation and selection with identical items and candidate ordering. Test includes natural and AI-authored text.

All systems share 100 ILS, including previous attempts. Changing selection does not reset spending. Offline checks are not live Colab or account validation. See [the operating guide](../docs/reproducible_evaluation.md). Historical outputs are preserved; no retraining or paid runtime purchase occurs here.

## 1. Install the identified source

Fill `CODE_REVISION` with the reviewed 40-character commit. For the default archive route, upload its `git archive --format=tar` file with Colab's Files panel, set `ARCHIVE_PATH` and copy the independently supplied SHA-256. This works without publishing the branch. The optional `git` route fetches only the exact commit; it requires that commit to be accessible on the remote. The package is installed normally so the running kernel can import it immediately. After changing commits or if Colab reports already-loaded dependency changes, restart the runtime and rerun setup before continuing.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
import urllib.request

CODE_REVISION = ""  # Reviewed 40-character commit; never main/HEAD/a branch.
SOURCE_MODE = "archive"  # "archive" or "git"
ARCHIVE_PATH = Path("/content/evaluation-release.tar")
ARCHIVE_SHA256 = ""  # Supplied with the release, not inferred from the upload.
REPOSITORY_URL = "https://github.com/BenCarmel123/hebrew-acronym-disambiguation.git"
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise ValueError("Set the reviewed full commit SHA before installation")
REPO_DIR = Path("/content/hebrew-eval-" + CODE_REVISION)
REPO_DIR.mkdir(parents=True, exist_ok=True)
if SOURCE_MODE == "archive":
    if not re.fullmatch(r"[0-9a-f]{64}", ARCHIVE_SHA256):
        raise ValueError("Set the release archive SHA-256")
    archive_bytes = ARCHIVE_PATH.read_bytes()
    if hashlib.sha256(archive_bytes).hexdigest() != ARCHIVE_SHA256:
        raise ValueError("Uploaded archive differs from the reviewed release")
    archive_revision = subprocess.check_output(
        ["git", "get-tar-commit-id"], input=archive_bytes).decode().strip()
    if archive_revision != CODE_REVISION:
        raise ValueError("Archive commit differs from CODE_REVISION")
    with tarfile.open(ARCHIVE_PATH, "r:") as archive:
        archive.extractall(REPO_DIR, filter="data")
elif SOURCE_MODE == "git":
    subprocess.run(["git", "init", str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1",
                    REPOSITORY_URL, CODE_REVISION], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
    actual = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
    if actual != CODE_REVISION:
        raise ValueError("Fetched source differs from CODE_REVISION")
else:
    raise ValueError("Choose archive or git")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_DIR)], check=True)
# Regular installation is immediately importable in this running notebook kernel.
# Check this project's pins; Colab can contain unrelated preinstalled conflicts.
for line in (REPO_DIR / "requirements.txt").read_text().splitlines():
    requirement = line.strip()
    if requirement and not requirement.startswith("#"):
        package_name, required_version = requirement.split("==", 1)
        if importlib.metadata.version(package_name) != required_version:
            raise RuntimeError(f"Required package version mismatch: {package_name}")
pip_report = subprocess.run([sys.executable, "-m", "pip", "check"],
                            capture_output=True, text=True)
if pip_report.returncode:
    print("Colab environment dependency report (inspect separately from project pins):")
    print(pip_report.stdout or pip_report.stderr)
print("Installed source commit:", CODE_REVISION)

## 2. Connect persistent storage and select systems

Use the same `RUN_NAME` after a disconnect or when adding a system. For this new code release use a fresh name; retain the older folder and include **all earlier costs, including failed/uncertain Gemini attempts**, in `PRIOR_SPEND_ILS`. This required value starts unset. Changing it within a session is rejected.

Set `SELECTED_SYSTEMS` to any subset of the four approved systems. Add only their API keys to Colab Secrets, with notebook access. Missing keys are reported and do not prevent other systems from running. Never place keys in cells or chat. Full-test eligibility is checked separately for each system.

In [ ]:
from google.colab import drive, userdata
import importlib.metadata
import platform
import torch
from hebrew_acronyms.staged_evaluation import (
    SYSTEM_NAMES, prepare_session, run_system_pilot, review_system_pilot,
    run_system_full, session_summary,
)
from hebrew_acronyms.test_evaluation import summarize_evaluation
from hebrew_acronyms.models.qwen.eval import inspect_ollama
from hebrew_acronyms.models.openai.eval import inspect_openai_model
from hebrew_acronyms.models.anthropic.eval import inspect_anthropic_model
from hebrew_acronyms.models.common.http import request_json

RUN_NAME = "evaluation-staged-20261009-01"
SELECTED_SYSTEMS = ["qwen", "gemini", "openai", "anthropic"]
PRIOR_SPEND_ILS = None  # REQUIRED: earlier costs/allowances outside this session.
ILS_PER_USD_ALLOWANCE = 4.0  # Conversion incl. fee/tax margin, not quoted FX.
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("Use a simple stable run name")
if not SELECTED_SYSTEMS or len(set(SELECTED_SYSTEMS)) != len(SELECTED_SYSTEMS) or not set(SELECTED_SYSTEMS) <= set(SYSTEM_NAMES):
    raise ValueError("Select one or more of the four approved systems, without duplicates")
if PRIOR_SPEND_ILS is None:
    raise ValueError("Account for earlier attempts in PRIOR_SPEND_ILS before proceeding")
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/NLP/evaluation-runs") / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
AVAILABILITY = {name: {"status": "not_selected"} for name in SYSTEM_NAMES}
SECRET_NAMES = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
for name in SELECTED_SYSTEMS:
    if name not in SECRET_NAMES:
        continue
    secret_name = SECRET_NAMES[name]
    os.environ.pop(secret_name, None)  # Do not reuse a stale key after a failed Secrets read.
    try:
        secret_value = userdata.get(secret_name)
        if not secret_value:
            raise ValueError("empty")
        os.environ[secret_name] = secret_value
        AVAILABILITY[name] = {"status": "credentials_loaded"}
        del secret_value
    except Exception:
        AVAILABILITY[name] = {"status": "unavailable", "reason": "missing_secret"}
ENVIRONMENT = {
    "code_revision": CODE_REVISION, "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ("torch", "transformers", "requests", "numpy")},
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "archive_sha256": ARCHIVE_SHA256 if SOURCE_MODE == "archive" else None,
}
with (OUTPUT_ROOT / f"environment-{time.time_ns()}.json").open("x") as stream:
    json.dump(ENVIRONMENT, stream, indent=2)
print(ENVIRONMENT)
print("Persistent outputs:", OUTPUT_ROOT)
print(AVAILABILITY)

## 3. Start and inspect Qwen when selected

Only selecting `qwen` installs Ollama and downloads `qwen2.5:7b`. Prefer a free GPU. Inspect the resolved digest, template, quantization and stops before its pilot. A setup failure is reported and other selected providers can continue. No model substitution is performed.

In [ ]:
QWEN_MODEL = "qwen2.5:7b"
QWEN_OPTIONS = {"temperature": 0, "seed": 42, "num_predict": 512}
OLLAMA_URL = "http://localhost:11434"
QWEN_IDENTITY = None
QWEN_STABLE_FIELDS = ("model", "digest", "server_version", "options",
                      "template", "model_parameters", "details", "model_info")
QWEN_FROZEN_IDENTITY = None
import requests
if "qwen" in SELECTED_SYSTEMS:
    try:
        if shutil.which("ollama") is None:
            subprocess.run(["apt-get", "-qq", "update"], check=True)
            subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)
            installer = Path("/content/ollama-install.sh")
            urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
            subprocess.run(["sh", str(installer)], check=True)
        import requests
        try:
            requests.get(OLLAMA_URL + "/api/version", timeout=5).raise_for_status()
        except requests.RequestException:
            ollama_log = (OUTPUT_ROOT / f"ollama-{time.time_ns()}.log").open("w")
            ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
            for _ in range(30):
                time.sleep(1)
                try:
                    requests.get(OLLAMA_URL + "/api/version", timeout=2).raise_for_status()
                    break
                except requests.RequestException:
                    pass
            else:
                raise RuntimeError("Ollama did not start; inspect its saved log")
        subprocess.run(["ollama", "pull", QWEN_MODEL], check=True)
        QWEN_IDENTITY = inspect_ollama(QWEN_MODEL, options=QWEN_OPTIONS)
        # Keep full session evidence outside the frozen run identity (pull timestamps vary).
        with (OUTPUT_ROOT / f"qwen-inspection-{time.time_ns()}.json").open("x") as stream:
            json.dump(QWEN_IDENTITY, stream, ensure_ascii=False, indent=2)
        QWEN_FROZEN_IDENTITY = {key: QWEN_IDENTITY[key] for key in QWEN_STABLE_FIELDS}
        AVAILABILITY["qwen"] = {"status": "available"}
        print(json.dumps(QWEN_FROZEN_IDENTITY, ensure_ascii=False, indent=2))
    except Exception:
        AVAILABILITY["qwen"] = {"status": "unavailable", "reason": "local_model_setup_failed"}
print(AVAILABILITY["qwen"])

## 4. Check selected accounts and freeze model settings

Lookups do not generate text or establish available credit. Failures are reported per system and do not authorize substitutes. Only available selected systems enter the pilot. Gemini uses low thinking; Claude uses adaptive thinking with low effort. Output caps include provider-billed thinking where applicable.

USD/million token assumptions checked 9 October 2026: [Gemini](https://ai.google.dev/gemini-api/docs/pricing), [OpenAI](https://developers.openai.com/api/docs/models/gpt-4.1-mini), [Claude](https://platform.claude.com/docs/en/models/haiku-5-5/overview). Verify tariffs before live execution; cached input is charged conservatively at the full rate.

In [ ]:
SYSTEMS = [
    {"name": "qwen", "provider": "qwen", "model": QWEN_MODEL,
     "settings": {"expected_digest": (QWEN_IDENTITY or {}).get("digest"), "base_url": OLLAMA_URL,
                  "timeout": 120, "options": QWEN_OPTIONS}},
    {"name": "gemini", "provider": "gemini", "model": "gemini-3.8-flash",
     "settings": {"timeout": 120, "generation_config": {
         "maxOutputTokens": 1024,
         "thinkingConfig": {"thinkingLevel": "low"}}}},
    {"name": "openai", "provider": "openai", "model": "gpt-4.1-mini-2025-04-14",
     "settings": {"timeout": 120, "max_output_tokens": 512, "temperature": 0}},
    {"name": "anthropic", "provider": "anthropic", "model": "claude-haiku-5-5",
     "settings": {"timeout": 120, "max_output_tokens": 1024, "effort": "low"}},
]
RATES = {"qwen": {"input": 0.0, "output": 0.0},
         "gemini": {"input": 0.75, "output": 3.75},
         "openai": {"input": 0.40, "output": 1.60},
         "anthropic": {"input": 0.10, "output": 0.50}}
for system in SYSTEMS:
    name = system["name"]
    if name not in SELECTED_SYSTEMS or name == "qwen" or AVAILABILITY[name]["status"] == "unavailable":
        continue
    if name == "openai":
        info = inspect_openai_model(model=system["model"])
    elif name == "anthropic":
        info = inspect_anthropic_model(model=system["model"])
    else:
        payload, info = request_json(
            requests.get, "https://generativelanguage.googleapis.com/v1beta/models/" + system["model"],
            provider="Gemini", secret=os.environ["GEMINI_API_KEY"], env_name="GEMINI_API_KEY",
            headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]}, timeout=30)
        if payload is not None and payload.get("name") == "models/" + system["model"] and "generateContent" in (payload.get("supportedGenerationMethods") or []):
            info["status"] = "available"
    # Save only allowlisted diagnostics, never upstream body/message or credentials.
    AVAILABILITY[name] = {key: info.get(key) for key in
        ("status", "http_status", "error_category", "provider_blocked", "retry_after")}
def available_selected_systems():
    # Read current notebook selection/readiness; cached lists cannot authorize calls.
    if not SELECTED_SYSTEMS or len(set(SELECTED_SYSTEMS)) != len(SELECTED_SYSTEMS) or not set(SELECTED_SYSTEMS) <= set(SYSTEM_NAMES):
        raise ValueError("Select approved systems without duplicates")
    return [s for s in SYSTEMS if s["name"] in SELECTED_SYSTEMS
            and AVAILABILITY.get(s["name"], {}).get("status") == "available"]
with (OUTPUT_ROOT / f"availability-{time.time_ns()}.json").open("x") as stream:
    json.dump(AVAILABILITY, stream, indent=2)
print(json.dumps(AVAILABILITY, indent=2))
DEV_PATH = REPO_DIR / "data/splits/dev_items.csv"
TEST_PATH = REPO_DIR / "data/splits/test_items.csv"
def model_identity(system):
    return {key: QWEN_IDENTITY[key] for key in QWEN_STABLE_FIELDS} if system["name"] == "qwen" else None

## 5. Run or resume each selected development pilot

Each system receives the same first ten development rows and both tasks (20 logical responses). Candidate orders derive from the same seed and item identities. Each request and outcome is persisted. Two attempts per request are allowed; explicit provider blocks stop further requests, transient waiting is bounded, and ambiguous interrupted requests are never resent automatically.

All saved pilot/full journals contribute to the same 100 ILS ceiling. The per-call reserve includes uncertain usage. Keep every system directory when changing selection or resuming; never run simultaneous writers.

In [ ]:
# Upper input allowance: prompt UTF-8 bytes plus provider-envelope margin.
from hebrew_acronyms.models.common.pairs import load_rows
from hebrew_acronyms.models.common.eval import build_generate_prompt, build_select_prompt
all_rows = load_rows(DEV_PATH) + load_rows(TEST_PATH)
max_prompt_bytes = max(len(prompt.encode("utf-8")) for row in all_rows for prompt in (
    build_generate_prompt(row["acronym"], row["sentence"]),
    build_select_prompt(row["acronym"], row["sentence"], row["candidates"].split("|")),
))
input_token_allowance = max_prompt_bytes + 1024
if input_token_allowance >= 100_000:
    raise ValueError("Recheck long-context price tier before any calls")
OUTPUT_CAPS = {"qwen": 512, "gemini": 1024, "openai": 512, "anthropic": 1024}
RESERVE_PER_CALL_USD = {
    name: (input_token_allowance * rate["input"] + OUTPUT_CAPS[name] * rate["output"]) / 1_000_000
    for name, rate in RATES.items()
}
def current_session():
    return prepare_session(
        OUTPUT_ROOT, DEV_PATH, TEST_PATH, code_revision=CODE_REVISION,
        prior_spend_ils=PRIOR_SPEND_ILS, rates=RATES, reserves=RESERVE_PER_CALL_USD,
        ils_per_usd=ILS_PER_USD_ALLOWANCE,
    )
current_session()
for system in available_selected_systems():
    result = run_system_pilot(OUTPUT_ROOT, system, model_identity=model_identity(system))
    print(system["name"], result["manifest"]["identity_sha256"],
          {key: result["summary"][key] for key in ("n_calls", "n_completed", "n_pending", "n_ambiguous")})
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))

## 6. Inspect each pilot and its projection

Inspect every response, identity, finish reason, truncation and usage. Semantic mistakes remain observations; pilot eligibility never requires a high score. Each complete 20-response pilot receives its own projection from 10 to 395 items plus 25% reserve, with all other systems' spending included.

Copy a pilot's printed `pilot_identity` into `INSPECTED_PILOTS` in the next cell **only after inspecting its records**. Incomplete/blocked systems remain explicitly reported; available systems may advance independently.

In [ ]:
current_session()  # Revalidate even when preceding notebook cells were skipped.
for system in available_selected_systems():
    name = system["name"]
    pilot_summary = summarize_evaluation(OUTPUT_ROOT / name / "dev-pilot")
    print(name, json.dumps(pilot_summary["by_system_task"], ensure_ascii=False))
    for record in pilot_summary["records"]:
        print(json.dumps(record, ensure_ascii=False))
    try:
        review = review_system_pilot(OUTPUT_ROOT, system, model_identity=model_identity(system))
        print(name, "pilot_identity:", review["pilot_identity"])
        print(json.dumps(review["cost"], indent=2))
        print("Projected full minutes:", round(pilot_summary["timings_seconds"][name] * 395 / 10 / 60, 1))
        print("Projected total ILS including all saved systems:", review["projected_total_ils"])
    except ValueError:
        print(name, "Full test gated: pilot incomplete or identity/configuration mismatch; inspect saved evidence.")
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))

## 7. Run or resume eligible systems on the full test

Fill `INSPECTED_PILOTS` with exact hashes printed above, for example `{"openai": "<pilot_identity>"}`. All 395 test items and both tasks are retained (790 logical responses per system). Before each system starts, the package reloads its pilot, validates current settings/source/model identity, checks complete coverage, and recomputes the projection and total spending from every journal. A stale inspection cannot authorize a changed pilot.

Adding a system requires its own successful pilot. Unselected, uninspected or blocked systems are reported explicitly. Repeating this cell preserves completed responses, uncertain charges and the shared cap. A code/configuration change needs a new named session with previous costs carried forward.

In [ ]:
INSPECTED_PILOTS = {}  # system name -> exact inspected pilot identity SHA-256
current_session()  # Revalidate even when preceding notebook cells were skipped.
for system in available_selected_systems():
    name = system["name"]
    if name not in INSPECTED_PILOTS:
        print(name, "Full test not run: pilot not inspected")
        continue
    try:
        test_summary = run_system_full(
            OUTPUT_ROOT, system, inspected_pilot_identity=INSPECTED_PILOTS[name],
            model_identity=model_identity(system),
        )
        print(name, {key: test_summary[key] for key in
              ("n_items", "n_records", "n_calls", "n_completed", "n_pending", "n_ambiguous")})
    except ValueError:
        print(name, "Full test gated: verify saved pilot identity, coverage, inputs and shared budget.")
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))

## 8. Preserve the baseline comparison and checkpoint route

The deterministic baselines are retained: uniform-random expectation, most-frequent, most-mined and oracle. The trained DictaBERT and untrained similarity baseline remain separate systems; no retraining occurs here. Checkpoint reuse must identify the full weights and tokenizer/config snapshot. Historical training seed, selected epoch and exact training data are not recovered from a state dictionary.

The existing package [Colab checkpoint adapter](../src/hebrew_acronyms/models/dictabert_cross_encoder/colab.py) accepts the authorized weights SHA-256 `23bbff0324a7ce4d9d4a24a9ebfb87a4f6ac296bf1f1ce1bd25126c09260267f`. Supply an existing local snapshot and checkpoint from Drive. Do not use the old notebook's inferred seed or mutable model download as training provenance. The [checkpoint guide](../docs/checkpoints.md) describes the preserved inference workflow.

**Encoder execution limitation:** the historical test CSV lacks explicit target spans required by the strict package encoder. Preserve these arms, but qualify the target mapping before fresh checkpoint evaluation; do not silently take the first matching occurrence. Historical encoder results remain separately identifiable. This notebook does not claim new encoder predictions.

In [ ]:
from hebrew_acronyms.test_baselines import run_test_baselines
baseline_summary = run_test_baselines(
    TEST_PATH, REPO_DIR / "data/mined/candidate_table.csv",
    OUTPUT_ROOT / "baselines", code_revision=CODE_REVISION,
)
print(baseline_summary["summary"])

## 9. Inspect and export saved evidence

`manifest.json` identifies code, inputs, complete prompts, candidate orders and settings. `attempts.jsonl` preserves attempts; `summary.json` includes coverage and raw records. Keep the complete Drive folder. Missing/incomplete responses stay in the denominator. Automatic letter parsing and historical string scoring are not human generation judgments.

The methods/analysis entry point remains [experimental_study.ipynb](experimental_study.ipynb). Do not copy a provisional aggregate into the manuscript: use an explicitly selected run and retain source groups, model versions and human-judgment provenance. The current dev paper exporter is not a full-test exporter. Downloading the saved archive below makes a local copy; it does not publish results or send them to another person.

In [ ]:
from google.colab import files
print(json.dumps(session_summary(OUTPUT_ROOT), ensure_ascii=False, indent=2))
for name in SYSTEM_NAMES:
    directory = OUTPUT_ROOT / name / "full-test"
    if (directory / "manifest.json").exists():
        saved_summary = summarize_evaluation(directory)
        print(json.dumps(saved_summary["by_system_task"], ensure_ascii=False, indent=2))
archive_path = shutil.make_archive("/content/" + RUN_NAME + "-" + str(time.time_ns()), "zip", OUTPUT_ROOT)
files.download(archive_path)